In [13]:
import csv
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.preprocessing import StandardScaler

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [14]:
# read data

df = pd.read_csv('src/breast_cancer_diagnostic.data', skiprows=2, header=None)

X = df.iloc[:, :-1] # everything except last column
y = df.iloc[:, -1] # last col

In [15]:

# using state=42 makes result reproducable. remove for random
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42
)

# scale the values
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# verify shape 18-12
print(X_train.shape)
print(X_test.shape)

(341, 31)
(228, 31)


In [16]:
y_train = y_train.values
y_test = y_test.values

X_train = np.hstack((np.ones((X_train.shape[0], 1)), X_train))
X_test = np.hstack((np.ones((X_test.shape[0], 1)), X_test))


print(y_train.shape)
print(X_train.shape)

(341,)
(341, 32)


In [17]:
# need a weight for all inputs BP Cholesterol Age Pregnant (w1,w2,w3,w4)
# need w0 (bias)
# score = w1*BP + w2*Cholesterol + w3*Age + w4*Pregnant + b

prev_loss = 0
current_loss = 0
threshold = 1e-4

fail_safe = 1000
iterations = 0
learning_rate = 0.01

weights = np.zeros(X_train.shape[1])

while iterations < fail_safe:
    # 1. predict
    z = np.dot(X_train, weights)
    y_pred = sigmoid(z)

    # 2. calculate loss
    m = len(y_train)
    epsilon = 1e-15
    y_pred = np.clip(y_pred, epsilon, 1 - epsilon)

    current_loss = -(1/m) * np.sum(
        y_train * np.log(y_pred) + (1 - y_train) * np.log(1 - y_pred)
    )

    # 3. stop if converged
    if iterations > 0 and abs(current_loss - prev_loss) < threshold:
        print(f"Iterations: {iterations} | current_loss: {current_loss}")
        break

    # 4. check error
    error = y_pred - y_train

    # 5. adjust weight
    gradient = np.dot(X_train.T, error) / m
    weights = weights - learning_rate * gradient

    # 6. update for next loop
    prev_loss = current_loss
    iterations += 1

    print(f"Iterations: {iterations} | current_loss: {current_loss}")


Iterations: 1 | current_loss: 0.6931471805599453
Iterations: 2 | current_loss: 0.6620585393486802
Iterations: 3 | current_loss: 0.6321681749298191
Iterations: 4 | current_loss: 0.6034004782812878
Iterations: 5 | current_loss: 0.5756820359754222
Iterations: 6 | current_loss: 0.5489425125884407
Iterations: 7 | current_loss: 0.5231152567713365
Iterations: 8 | current_loss: 0.498137659480248
Iterations: 9 | current_loss: 0.4739513081035076
Iterations: 10 | current_loss: 0.4505019844330473
Iterations: 11 | current_loss: 0.42773955126949526
Iterations: 12 | current_loss: 0.40561776549081807
Iterations: 13 | current_loss: 0.38409404729690283
Iterations: 14 | current_loss: 0.3631292276551821
Iterations: 15 | current_loss: 0.3426872894628623
Iterations: 16 | current_loss: 0.3227351128136469
Iterations: 17 | current_loss: 0.3032422309227655
Iterations: 18 | current_loss: 0.2841806005125437
Iterations: 19 | current_loss: 0.26552438855511584
Iterations: 20 | current_loss: 0.24724977599413844
Itera

In [18]:
# run the algo again but on test data
z_test = np.dot(X_test, weights)
y_prob = sigmoid(z_test)

# change probability into class label. Threshold is 0.5
y_pred = (y_prob >= 0.5).astype(int)

# compare accuracy
accuracy = np.mean(y_pred == y_test)

print(f"Accurary: {accuracy}")

Accurary: 0.14912280701754385


In [19]:
print(f"""
weights: 
    b: {weights[0]}
    BP: {weights[1]}
    Cholesterol: {weights[2]}
    Age: {weights[3]}
    Pregnant: {weights[4]}

""")


weights: 
    b: 8.627146515182377
    BP: 0.02615876969870725
    Cholesterol: -0.7324947794818535
    Age: -0.5870586194438399
    Pregnant: -0.727805960719902


